# 🌱 Generador de Dataset Agrícola para SLM

Este Colab genera automáticamente un dataset de conversaciones en lenguaje simple (estilo campesino colombiano) para entrenar un modelo de lenguaje pequeño (SLM) orientado a buenas prácticas agrícolas.

## ¿Cómo funciona?
1. **Lee un archivo JSON** con la lista de chunks ya extraídos y enriquecidos con metadatos
2. **Por cada chunk**, le pide a DeepSeek que genere una conversación en el formato requerido
3. **Guarda las conversaciones** en un dataset `.jsonl` acumulativo
4. **Lleva estadísticas** por cultivo
5. **Permite descargar** el dataset al finalizar
6. **Soporta reanudar** la generación si se interrumpe, o ampliar un dataset existente

---
## Estructura del archivo JSON de entrada
```json
{
  "chunks": [
    {
      "document_id": "Nombre del documento",
      "chunk_number": 1,
      "citation": "Referencia bibliográfica del documento",
      "date": "Fecha del documento",
      "text": "Contenido completo del chunk..."
    },
    {
      "document_id": "Nombre del documento",
      "chunk_number": 2,
      "citation": "Referencia bibliográfica del documento",
      "date": "Fecha del documento",
      "text": "Contenido completo del chunk..."
    }
  ]
}
```

---
## Estructura de cada entrada del dataset generado
```json
{
  "messages": [
    {"role": "system",    "content": "..."},
    {"role": "user",      "content": "<knowledge>\n...\n</knowledge>\n\nPregunta..."},
    {"role": "assistant", "content": "<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>"}
  ]
}
```

---
## 📦 Sección 1 — Instalación de dependencias
Instalamos las librerías necesarias para comunicarnos con la API de DeepSeek y mostrar el progreso.

In [ ]:
# Instalamos las librerías necesarias:
#   openai : cliente compatible con la API de DeepSeek
#   tqdm   : para mostrar barras de progreso

import sys
if 'google.colab' in sys.modules:
    get_ipython().system("pip install openai tqdm --quiet")
    print("✅ Dependencias instaladas correctamente.")
else:
    print("Entorno local/VPS. Dependencias se asumen instaladas a través de install_deps.")

---
## ⚙️ Sección 2 — Configuración
Aquí defines los parámetros principales: dónde está el JSON de chunks, tu API key de DeepSeek, el modelo a usar, etc.

> **Importante:** Nunca compartas tu API key públicamente.

In [ ]:
import os

# =============================================================
# ⚙️ PARÁMETROS DE CONFIGURACIÓN — Modifica estos según tu caso
# =============================================================

# --- API de DeepSeek ---
OPENAI_API_KEY = "sk-..."          # 🔑 Reemplaza con tu API key de DeepSeek
OPENAI_MODEL   = "deepseek-chat"   # Modelo a usar (deepseek-chat, deepseek-reasoner, etc.)

# --- Archivo JSON de entrada ---
# Ruta al JSON que contiene todos los chunks organizados por fuente.
# Puede estar en Drive o subirse directamente a /content/.
# Formatos soportados:
#   · Diccionario  → {"fuente_1": ["chunk1", "chunk2", ...], "fuente_2": [...], ...}
#   · Lista de listas → [["chunk1", "chunk2", ...], ["chunk1", ...], ...]
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    CHUNKS_JSON_FILE = "/content/drive/MyDrive/chunks_agricolas.json"
    OUTPUT_FILE   = "/content/drive/MyDrive/dataset_agricola.jsonl"
    PROGRESS_FILE = "/content/drive/MyDrive/dataset_progreso.json"
    STATS_FILE    = "/content/drive/MyDrive/dataset_estadisticas.json"
else:
    CHUNKS_JSON_FILE = "chunks_agricolas.json"
    OUTPUT_FILE   = "dataset_agricola.jsonl"
    PROGRESS_FILE = "dataset_progreso.json"
    STATS_FILE    = "dataset_estadisticas.json"

# --- Control del proceso ---
MAX_RETRIES   = 3     # Intentos máximos si la API falla en un chunk
DELAY_SECONDS = 1.5   # Pausa entre llamadas a la API (en segundos)

print("✅ Configuración cargada.")
print(f"   📄 Archivo de chunks : {CHUNKS_JSON_FILE}")
print(f"   💾 Archivo de salida : {OUTPUT_FILE}")
print(f"   🤖 Modelo DeepSeek   : {OPENAI_MODEL}")

---
## 🔗 Sección 3 — Montar Google Drive e importar librerías
Montamos Google Drive para acceder al JSON de chunks y a los archivos de salida.

In [ ]:
# Montar Google Drive (Solo en Google Colab)
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

# Importaciones generales
import json
import time
import re
from pathlib import Path
from collections import defaultdict

# Librerías de terceros
from openai import OpenAI
from tqdm.notebook import tqdm

# Inicializar el cliente de DeepSeek (compatible con el SDK de OpenAI)
client = OpenAI(api_key=OPENAI_API_KEY, base_url="https://api.deepseek.com")

print("✅ Google Drive montado e importaciones listas.")

---
## 📂 Sección 4 — Carga del archivo JSON de chunks
Esta función lee el JSON de entrada, valida su estructura y retorna la lista de chunks
listos para procesar. Cada chunk es un diccionario con los campos:
`document_id`, `chunk_number`, `citation`, `date` y `text`.

In [ ]:
def load_chunks_from_json(json_path):
    """
    Carga el archivo JSON de chunks y retorna una lista de dicts, uno por chunk.

    Estructura esperada del JSON:
    {
      "chunks": [
        {
          "document_id": "Nombre del documento",
          "chunk_number": 1,
          "citation":     "Referencia bibliográfica",
          "date":         "Fecha del documento",
          "text":         "Contenido completo del chunk..."
        },
        ...
      ]
    }

    Retorna una lista de los mismos dicts, descartando los que tengan un campo
    'text' vacío o con menos de 50 palabras.
    """
    json_file = Path(json_path)
    if not json_file.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo '{json_path}'.\n"
            f"Verifica que la ruta es correcta y que el archivo está en Drive."
        )

    with open(json_file, "r", encoding="utf-8") as f:
        raw = json.load(f)

    # Validar que el JSON tenga la clave 'chunks'
    if not isinstance(raw, dict) or "chunks" not in raw:
        raise ValueError(
            "El JSON no tiene el formato esperado. "
            "Debe ser un objeto con una clave 'chunks' que contenga una lista."
        )

    all_chunks = raw["chunks"]
    if not isinstance(all_chunks, list):
        raise ValueError("El campo 'chunks' debe ser una lista de objetos.")

    # Campos obligatorios en cada chunk
    required_fields = {"document_id", "chunk_number", "text"}

    valid_chunks    = []
    skipped_missing = 0   # Chunks con campos obligatorios ausentes
    skipped_short   = 0   # Chunks con texto demasiado corto

    for item in all_chunks:
        # Verificar campos obligatorios
        if not required_fields.issubset(item.keys()):
            skipped_missing += 1
            continue

        text = item.get("text", "")
        if not isinstance(text, str) or len(text.split()) < 50:
            skipped_short += 1
            continue

        valid_chunks.append(item)

    # ── Reporte de carga ──────────────────────────────────────────────────
    # Contamos cuántos documentos únicos hay
    doc_ids = {c["document_id"] for c in valid_chunks}

    print(f"✅ JSON cargado correctamente.")
    print(f"   🧩 Total de chunks en el archivo : {len(all_chunks)}")
    print(f"   ✅ Chunks válidos a procesar     : {len(valid_chunks)}")
    print(f"   ⚠️  Descartados (campos faltantes): {skipped_missing}")
    print(f"   ⚠️  Descartados (texto muy corto) : {skipped_short}")
    print(f"   📚 Documentos únicos detectados  : {len(doc_ids)}")
    print()
    for doc_id in sorted(doc_ids):
        n = sum(1 for c in valid_chunks if c["document_id"] == doc_id)
        print(f"   → {doc_id}: {n} chunks")

    return valid_chunks


def generate_chunk_id(document_id, chunk_number):
    """
    Genera un identificador único por chunk para el sistema de progreso.
    Usa el document_id y el chunk_number del propio JSON, que ya son únicos.
    Formato: 'Nombre del documento::chunk_0001'
    """
    return f"{document_id}::chunk_{int(chunk_number):04d}"


print("✅ Funciones de carga de chunks definidas.")

---
## 🤖 Sección 5 — Generación de conversaciones con DeepSeek
Aquí definimos el **prompt del sistema** con todas las reglas de estilo y contenido, y la función
que llama a la API para generar cada conversación a partir de un chunk de texto.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PROMPT DEL SISTEMA
#
# OPTIMIZACIÓN DE TOKENS:
# El LLM NO devuelve el chunk de texto en su respuesta. Solo genera la
# pregunta del usuario y la respuesta del asistente. El Colab inyecta el
# chunk en la etiqueta <knowledge> por código, una vez recibida la respuesta.
# Esto evita pagar los tokens del chunk en la salida del LLM, reduciendo
# el costo por iteración de forma significativa.
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """\
Eres un experto en agricultura colombiana con amplio conocimiento del campo y de cómo hablan los campesinos del país.
Tu tarea es generar conversaciones de entrenamiento para un asistente de inteligencia artificial enfocado en buenas prácticas agrícolas.

A partir del fragmento de texto que te proporcionaré, deberás hacer UNA de estas dos cosas:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OPCIÓN A — El fragmento NO es útil o relevante
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Si el fragmento de texto es:
  - Un índice, tabla de contenido, lista de referencias o bibliografía
  - Texto incompleto o incomprensible sin contexto
  - Información que no tiene nada que ver con prácticas agrícolas
  - Fragmento demasiado corto o sin información accionable

Responde ÚNICAMENTE con este JSON (y nada más):
{"skip": true, "reason": "explicación breve de por qué se omite"}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OPCIÓN B — El fragmento SÍ es útil y relevante
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Genera una conversación de entrenamiento en el siguiente formato JSON exacto:

{
  "messages": [
    {
      "role": "system",
      "content": "Eres un asistente inteligente. Analiza la información proporcionada en la etiqueta <knowledge> para responder a la solicitud del usuario. Responde estrictamente en el siguiente formato:\\n<reasoning>\\n...\\n</reasoning>\\n<answer>\\n...\\n</answer>"
    },
    {
      "role": "user",
      "content": "[AQUÍ VA ÚNICAMENTE LA PREGUNTA DEL USUARIO, sin ninguna etiqueta adicional]"
    },
    {
      "role": "assistant",
      "content": "<reasoning>\\n[RAZONAMIENTO DEL MODELO BASADO EN EL KNOWLEDGE]\\n</reasoning>\\n<answer>\\n[RESPUESTA FINAL]\\n</answer>"
    }
  ],
  "metadata": {
    "cultivo": "[nombre del cultivo principal, o 'general' si aplica a varios]",
    "tema": "[tema principal: ej. plagas, fertilización, riego, cosecha, enfermedades, siembra, postcosecha, etc.]"
  }
}

IMPORTANTE sobre el campo "user":
  - Escribe SOLO la pregunta del campesino, en texto plano.
  - NO incluyas la etiqueta <knowledge> ni el fragmento de texto.
  - NO escribas nada más que la pregunta.
  - El sistema añadirá el fragmento de contexto automáticamente.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
REGLAS OBLIGATORIAS para generar la conversación
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. LENGUAJE CAMPESINO COLOMBIANO:
   - Escribe como hablaría un campesino colombiano: sencillo, directo, sin rodeos.
   - Usa palabras del campo colombiano cuando aplique (ej. "matas" en vez de "plantas", "rozar" en vez de "desyerbar", "parcela" en vez de "lote").
   - El usuario pregunta de manera natural, sin usar vocabulario académico.
   - La respuesta también suena natural, como si viniera de un vecino del campo con experiencia.

2. EVITAR TECNICISMOS:
   - No uses palabras técnicas si hay una alternativa sencilla.
   - Si DEBES usar un término técnico, explícalo entre paréntesis con palabras simples.
   - Ejemplo correcto: "la antracnosis (una enfermedad que pudre los frutos)"

3. NO MENCIONAR LA FUENTE:
   - NUNCA digas "en el documento", "según el manual", "el texto dice", "los expertos recomiendan que..."
   - La respuesta debe sonar como conocimiento propio, ganado en el campo.

4. NO PEDIR DATOS ESPECIALIZADOS:
   - No preguntes cosas como: pH del suelo, altitud exacta, composición química del agua, etc.
   - SÍ puedes preguntar: ¿Llueve mucho donde vive? ¿El clima es frío o caliente? ¿La tierra es muy arcillosa (como barro) o suelta?

5. RESPUESTAS CONCRETAS:
   - No des consejos vagos. Sé específico.
   - MAL: "Debe mejorar la frecuencia de riego."
   - BIEN: "Riegue las matas cada dos días en verano, y cuando llueva seguido, deje de regar por unos tres o cuatro días para que la tierra no se encharque."

6. NÚMEROS Y CANTIDADES:
   - Evita porcentajes y fórmulas complejas.
   - Usa comparaciones simples: "un puño de abono", "una cucharada por palada de tierra", "cada quince días".
   - Si el dato numérico es importante, exprésalo de forma comprensible: en vez de "aplique 2.5 kg/ha", di "como un kilo de abono por cada pedazo de tierra del tamaño de una cancha de fútbol".

7. PRODUCTOS Y DÓNDE CONSEGUIRLOS:
   - Si recomiendas un producto (abono, fungicida, semilla, etc.), menciona que se puede encontrar en Colombia, idealmente en agroservicios o almacenes agropecuarios de la región.

8. RAZONAMIENTO (reasoning):
   - El reasoning DEBE mostrar cómo el modelo analiza el conocimiento del contexto para construir la respuesta.
   - El reasoning puede estar escrito de forma más técnica (es para el entrenamiento interno), pero la <answer> siempre debe ser sencilla.

9. FORMATO DE SALIDA:
   - Responde ÚNICAMENTE con el JSON. Sin texto antes ni después. Sin bloques de código markdown (``` no).
"""


print("✅ Prompt del sistema definido.")

In [ ]:
def generate_conversation(chunk_text, retries=MAX_RETRIES):
    """
    Envía un chunk de texto a la API de DeepSeek y genera una conversación
    de entrenamiento en el formato requerido.

    OPTIMIZACIÓN DE TOKENS:
    El LLM solo genera la pregunta del usuario y la respuesta del asistente.
    NO devuelve el chunk en su salida. Una vez recibida la respuesta, esta
    función inyecta el chunk_text en la etiqueta <knowledge> del campo
    "user" por código, antes de retornar la conversación final.

    Retorna:
      (dict, None)  → conversación completa lista para guardar en el dataset
      (None, str)   → chunk omitido o error; el str explica la razón
    """
    user_message = (
        f"Genera una conversación de entrenamiento basada en el siguiente "
        f"fragmento de texto de un manual agrícola:\n\n"
        f"FRAGMENTO:\n{chunk_text}"
    )

    for attempt in range(1, retries + 1):
        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_message}
                ],
                stream=False,
                temperature=0.75,
                max_tokens=2500,
                extra_body={"thinking": {"type": "disabled"}}
            )

            raw_content = response.choices[0].message.content.strip()

            # Limpieza: quitamos bloques de código markdown si el modelo los incluyó
            raw_content = re.sub(r'''(?m)^```(?:json)?\s*''', '', raw_content)
            raw_content = re.sub(r'''(?m)```\s*$''',          '', raw_content)
            raw_content = raw_content.strip()

            # Parseamos el JSON de la respuesta
            result = json.loads(raw_content)

            # Si el modelo decidió saltar este chunk, retornamos None
            if result.get("skip") is True:
                return None, result.get("reason", "chunk irrelevante")

            # Verificación básica de estructura
            if "messages" not in result or len(result["messages"]) < 3:
                raise ValueError("El JSON no tiene la estructura de mensajes esperada.")

            # ── Inyección del chunk en <knowledge> ───────────────────────────────
            # El LLM solo devolvió la pregunta en user.content (texto plano).
            # Aquí el Colab construye el campo completo por código:
            # envuelve la pregunta con la etiqueta <knowledge> y el chunk.
            # Esto ahorra todos los tokens del chunk en la salida del LLM.
            question = result["messages"][1]["content"]
            result["messages"][1]["content"] = (
                f"<knowledge>\n{chunk_text}\n</knowledge>\n\n{question}"
            )
            # ─────────────────────────────────────────────────────────────────────

            return result, None  # ✅ Éxito

        except json.JSONDecodeError as e:
            print(f"   ⚠️ Intento {attempt}/{retries} — Error al parsear JSON: {e}")
        except Exception as e:
            print(f"   ⚠️ Intento {attempt}/{retries} — Error en la API: {e}")

        if attempt < retries:
            wait = DELAY_SECONDS * attempt * 2
            print(f"   ⏳ Esperando {wait:.1f}s antes del siguiente intento...")
            time.sleep(wait)

    return None, "error irrecuperable tras múltiples intentos"


print("✅ Función de generación de conversaciones definida.")

---
## 💾 Sección 6 — Sistema de progreso y reanudación
Estas funciones permiten guardar el estado del proceso para que, si se interrumpe, puedas continuar desde donde quedaste. También permiten cargar un dataset existente para ampliarlo.

In [ ]:
def load_progress(progress_file):
    """
    Carga el archivo de progreso que registra qué chunk IDs ya fueron procesados.
    Si el archivo no existe, retorna un conjunto vacío.
    """
    progress_path = Path(progress_file)
    if progress_path.exists():
        with open(progress_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        processed_ids = set(data.get("processed_chunk_ids", []))
        print(f"📂 Progreso cargado: {len(processed_ids)} chunks ya procesados anteriormente.")
        return processed_ids
    else:
        print("🆕 No se encontró archivo de progreso. Se empezará desde el principio.")
        return set()


def save_progress(progress_file, processed_ids):
    """
    Guarda el conjunto de IDs de chunks ya procesados en el archivo de progreso.
    Se llama después de procesar cada chunk para que el estado siempre esté actualizado.
    """
    with open(progress_file, 'w', encoding='utf-8') as f:
        json.dump({"processed_chunk_ids": list(processed_ids)}, f, ensure_ascii=False)


def load_stats(stats_file):
    """
    Carga las estadísticas acumuladas de conversaciones por cultivo.
    Si el archivo no existe, retorna un defaultdict vacío.
    """
    stats_path = Path(stats_file)
    if stats_path.exists():
        with open(stats_path, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        stats = defaultdict(int, raw.get("por_cultivo", {}))
        print(f"📊 Estadísticas cargadas. Cultivos registrados: {dict(stats)}")
        return stats
    return defaultdict(int)


def save_stats(stats_file, stats_cultivo):
    """
    Guarda las estadísticas de conversaciones por cultivo en el archivo de estadísticas.
    """
    with open(stats_file, 'w', encoding='utf-8') as f:
        json.dump({"por_cultivo": dict(stats_cultivo)}, f, ensure_ascii=False, indent=2)


def append_conversation_to_dataset(output_file, conversation_dict):
    """
    Agrega una conversación al archivo de dataset en formato JSONL.
    JSONL = JSON Lines: cada línea es un objeto JSON independiente.
    Esto es el formato estándar para datasets de fine-tuning de LLMs.
    """
    with open(output_file, 'a', encoding='utf-8') as f:
        # Guardamos solo la parte 'messages' (sin los metadatos internos)
        entry = {"messages": conversation_dict["messages"]}
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')


def count_dataset_entries(output_file):
    """
    Cuenta cuántas conversaciones hay actualmente en el archivo de dataset.
    """
    output_path = Path(output_file)
    if not output_path.exists():
        return 0
    with open(output_path, 'r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())


print("✅ Funciones de progreso y persistencia definidas.")

---
## 🔄 Sección 7 — Opción de inicio: nuevo dataset o ampliar uno existente
Aquí eliges si quieres empezar desde cero o continuar/ampliar un dataset que ya existe.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# MODO DE INICIO
# Opciones:
#   "nuevo"     → Borra el progreso anterior y empieza desde cero
#   "continuar" → Retoma donde quedó (usa el archivo de progreso)
#   "ampliar"   → Carga un dataset y archivo de progreso existentes y agrega más
#
# Si usas "ampliar", asegúrate de que OUTPUT_FILE y PROGRESS_FILE ya existan.
# ─────────────────────────────────────────────────────────────────────────

MODO_INICIO = "nuevo"  # Cambia a "continuar" o "ampliar" según sea necesario

# ─────────────────────────────────────────────────────────────────────────
# Inicialización según el modo elegido
# ─────────────────────────────────────────────────────────────────────────

if MODO_INICIO == "nuevo":
    # Borramos los archivos anteriores si existen
    for f in [OUTPUT_FILE, PROGRESS_FILE, STATS_FILE]:
        if Path(f).exists():
            Path(f).unlink()
    processed_ids = set()
    stats_cultivo  = defaultdict(int)
    print("🆕 Modo NUEVO: Se empezará a generar el dataset desde cero.")

elif MODO_INICIO in ("continuar", "ampliar"):
    # Cargamos el progreso y estadísticas existentes
    processed_ids = load_progress(PROGRESS_FILE)
    stats_cultivo  = load_stats(STATS_FILE)
    existing_count = count_dataset_entries(OUTPUT_FILE)
    print(f"🔄 Modo {MODO_INICIO.upper()}: Dataset existente con {existing_count} conversaciones.")

else:
    raise ValueError(f"MODO_INICIO no válido: '{MODO_INICIO}'. Usa 'nuevo', 'continuar' o 'ampliar'.")

print(f"\n📌 Chunks ya procesados: {len(processed_ids)}")

---
## 🚀 Sección 8 — Proceso principal de generación
El bucle central del Colab. Carga la lista de chunks desde el JSON, itera sobre cada uno,
llama a DeepSeek para generar la conversación y guarda el resultado en el dataset.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# PROCESO PRINCIPAL DE GENERACIÓN
# ─────────────────────────────────────────────────────────────────────────

# Contadores de esta sesión
session_generated = 0   # Conversaciones nuevas generadas
session_skipped   = 0   # Chunks irrelevantes omitidos por el modelo
session_errors    = 0   # Chunks con error irrecuperable
session_resumed   = 0   # Chunks ya procesados en sesiones anteriores

# 1. Cargar todos los chunks desde el JSON
all_chunks = load_chunks_from_json(CHUNKS_JSON_FILE)

if not all_chunks:
    print("❌ No hay chunks válidos para procesar. Verifica el archivo JSON.")
else:
    print(f"\n{'='*60}")
    print(f" 🚀 INICIANDO GENERACIÓN DEL DATASET")
    print(f"{'='*60}\n")

    for chunk in tqdm(all_chunks, desc="Procesando chunks"):

        doc_id       = chunk["document_id"]
        chunk_number = chunk["chunk_number"]
        chunk_text   = chunk["text"]
        chunk_id     = generate_chunk_id(doc_id, chunk_number)

        # Saltamos chunks ya procesados en sesiones anteriores
        if chunk_id in processed_ids:
            session_resumed += 1
            continue

        # 2. Llamar a DeepSeek para generar la conversación
        conversation, skip_reason = generate_conversation(chunk_text)

        # 3. Procesar el resultado
        if conversation is not None:
            # ✅ Conversación generada con éxito
            metadata = conversation.get("metadata", {})
            cultivo  = metadata.get("cultivo", "general").lower().strip()

            append_conversation_to_dataset(OUTPUT_FILE, conversation)
            stats_cultivo[cultivo] += 1
            session_generated += 1

        elif skip_reason and "irrecuperable" in skip_reason:
            # ❌ Error que no pudo resolverse con reintentos
            session_errors += 1
            tqdm.write(f"   ❌ Error en {chunk_id}: {skip_reason}")

        else:
            # ⏭️ Chunk irrelevante, el modelo decidió omitirlo
            session_skipped += 1

        # 4. Marcar el chunk como procesado y persistir el estado
        processed_ids.add(chunk_id)
        save_progress(PROGRESS_FILE, processed_ids)
        save_stats(STATS_FILE, stats_cultivo)

        # 5. Pausa para no sobrecargar la API
        time.sleep(DELAY_SECONDS)

    # ─────────────────────────────────────────────────────────────────
    # RESUMEN FINAL DE LA SESIÓN
    # ─────────────────────────────────────────────────────────────────
    total = count_dataset_entries(OUTPUT_FILE)
    print(f"\n{'='*60}")
    print(f" 🎉 PROCESO COMPLETADO")
    print(f"{'='*60}")
    print(f"   ✅ Conversaciones generadas esta sesión  : {session_generated}")
    print(f"   ⏭️  Chunks irrelevantes omitidos          : {session_skipped}")
    print(f"   🔄 Chunks ya procesados (reanudados)    : {session_resumed}")
    print(f"   ❌ Chunks con error irrecuperable        : {session_errors}")
    print(f"   📊 Total de conversaciones en el dataset : {total}")
    print(f"{'='*60}")

---
## 📊 Sección 9 — Estadísticas del dataset
Muestra cuántas conversaciones hay por cultivo y cuáles son los temas más frecuentes.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# ESTADÍSTICAS POR CULTIVO
# ─────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print(" 📊 ESTADÍSTICAS DEL DATASET")
print("="*60)

total_conversations = count_dataset_entries(OUTPUT_FILE)
print(f"\n🌱 Total de conversaciones en el dataset: {total_conversations}")
print(f"\n📈 Distribución por cultivo:")
print("-" * 40)

if stats_cultivo:
    # Ordenamos de mayor a menor cantidad
    sorted_stats = sorted(stats_cultivo.items(), key=lambda x: x[1], reverse=True)
    max_count = sorted_stats[0][1] if sorted_stats else 1

    for cultivo, count in sorted_stats:
        # Barra de progreso visual simple
        bar_length = int((count / max_count) * 30)
        bar = "█" * bar_length
        print(f"   {cultivo.capitalize():<25} {bar} {count}")
else:
    print("   (No hay estadísticas disponibles aún)")

print("\n" + "="*60)

# ─────────────────────────────────────────────────────────────────────────
# ESTADÍSTICAS ADICIONALES: análisis del archivo JSONL
# ─────────────────────────────────────────────────────────────────────────

print("\n📋 Vista previa de la primera conversación del dataset:")
print("-" * 60)

output_path = Path(OUTPUT_FILE)
if output_path.exists() and output_path.stat().st_size > 0:
    with open(output_path, 'r', encoding='utf-8') as f:
        first_line = f.readline().strip()
    if first_line:
        first_entry = json.loads(first_line)
        # Mostramos el mensaje del usuario y del asistente (resumido)
        for msg in first_entry.get("messages", []):
            role = msg['role'].upper()
            content_preview = msg['content'][:300] + "..." if len(msg['content']) > 300 else msg['content']
            print(f"\n[{role}]\n{content_preview}")
            print("-" * 40)
else:
    print("   El archivo de dataset está vacío o no existe.")

---
## 🔍 Sección 10 — Validación del dataset
Verifica que todas las entradas del dataset tengan el formato correcto.

In [ ]:
def validate_dataset(output_file):
    """
    Valida que cada entrada del dataset JSONL tenga la estructura correcta:
    - Debe tener la clave 'messages'
    - Debe tener 3 mensajes: system, user, assistant
    - El mensaje del usuario debe contener <knowledge>
    - El mensaje del asistente debe contener <reasoning> y <answer>

    Retorna el número de entradas válidas e inválidas.
    """
    valid   = 0
    invalid = 0
    issues  = []

    output_path = Path(output_file)
    if not output_path.exists():
        print("❌ El archivo de dataset no existe.")
        return 0, 0

    with open(output_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
                msgs  = entry.get("messages", [])

                # Verificaciones
                assert len(msgs) == 3, f"Se esperan 3 mensajes, hay {len(msgs)}"
                assert msgs[0]['role'] == 'system',    "Mensaje 0 debe ser 'system'"
                assert msgs[1]['role'] == 'user',      "Mensaje 1 debe ser 'user'"
                assert msgs[2]['role'] == 'assistant', "Mensaje 2 debe ser 'assistant'"
                assert '<knowledge>' in msgs[1]['content'], "Falta <knowledge> en el mensaje del usuario"
                assert '<reasoning>' in msgs[2]['content'], "Falta <reasoning> en el mensaje del asistente"
                assert '<answer>'    in msgs[2]['content'], "Falta <answer> en el mensaje del asistente"

                valid += 1

            except (json.JSONDecodeError, AssertionError, KeyError) as e:
                invalid += 1
                issues.append(f"Línea {line_num}: {e}")

    print(f"\n🔍 VALIDACIÓN DEL DATASET")
    print("-" * 40)
    print(f"   ✅ Entradas válidas  : {valid}")
    print(f"   ❌ Entradas inválidas: {invalid}")

    if issues:
        print(f"\n   Problemas encontrados (primeros 10):")
        for issue in issues[:10]:
            print(f"   → {issue}")
    else:
        print("   🎉 ¡Todas las entradas tienen el formato correcto!")

    return valid, invalid


# Ejecutar la validación
valid_count, invalid_count = validate_dataset(OUTPUT_FILE)

---
## 📥 Sección 11 — Descarga del dataset
Descarga el archivo `.jsonl` del dataset y el resumen de estadísticas.

In [ ]:
import sys
import shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# Definimos los nombres de los archivos de descarga
download_dataset_path = "/content/dataset_agricola_final.jsonl" if IN_COLAB else "dataset_agricola_final.jsonl"
download_stats_path   = "/content/estadisticas_dataset.json" if IN_COLAB else "estadisticas_dataset.json"
download_summary_path = "/content/resumen_dataset.txt" if IN_COLAB else "resumen_dataset.txt"

# 1. Copiamos el dataset al directorio local para descargarlo
if Path(OUTPUT_FILE).exists():
    if OUTPUT_FILE != download_dataset_path:
        shutil.copy(OUTPUT_FILE, download_dataset_path)
    print(f"✅ Dataset copiado a: {download_dataset_path}")
else:
    print("❌ El archivo de dataset no existe. Ejecuta primero la Sección 8.")

# 2. Creamos un resumen de estadísticas en texto plano
with open(download_summary_path, 'w', encoding='utf-8') as f:
    f.write("RESUMEN DEL DATASET AGRÍCOLA\n")
    f.write("=" * 40 + "\n\n")
    f.write(f"Total de conversaciones: {count_dataset_entries(OUTPUT_FILE)}\n\n")
    f.write("Conversaciones por cultivo:\n")
    f.write("-" * 30 + "\n")

    sorted_stats = sorted(stats_cultivo.items(), key=lambda x: x[1], reverse=True)
    for cultivo, count in sorted_stats:
        f.write(f"  {cultivo.capitalize()}: {count}\n")

print(f"✅ Resumen creado en: {download_summary_path}")

# 3. Copiamos las estadísticas JSON
if Path(STATS_FILE).exists():
    if STATS_FILE != download_stats_path:
        shutil.copy(STATS_FILE, download_stats_path)
    print(f"✅ Estadísticas copiadas a: {download_stats_path}")

print("\n" + "="*60)
print(" 📥 DESCARGANDO ARCHIVOS...")
print("="*60)

# 4. Descargamos todos los archivos
if IN_COLAB:
    from google.colab import files
    if Path(download_dataset_path).exists():
        print("\n⬇️  Descargando el dataset principal (.jsonl)...")
        files.download(download_dataset_path)

    if Path(download_summary_path).exists():
        print("⬇️  Descargando el resumen de estadísticas (.txt)...")
        files.download(download_summary_path)

    if Path(download_stats_path).exists():
        print("⬇️  Descargando estadísticas detalladas (.json)...")
        files.download(download_stats_path)
    print("\n✅ Descarga completada.")
else:
    print("\n[INFO] Ejecución local completada. Los archivos se guardaron en:")
    print(f"  - Dataset principal: {download_dataset_path}")
    print(f"  - Resumen de estadísticas: {download_summary_path}")
    print(f"  - Estadísticas detalladas: {download_stats_path}")

---
## 🔧 Sección 12 — Herramientas adicionales
Celdas de utilidad para inspeccionar y gestionar el dataset.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# UTILIDAD: Ver una conversación específica del dataset
# ─────────────────────────────────────────────────────────────────────────

ENTRADA_A_VER = 0  # Cambia este número para ver otra entrada (0 = primera, 1 = segunda, etc.)

output_path = Path(OUTPUT_FILE)

if output_path.exists():
    with open(output_path, 'r', encoding='utf-8') as f:
        all_lines = [l.strip() for l in f if l.strip()]

    if ENTRADA_A_VER < len(all_lines):
        entry = json.loads(all_lines[ENTRADA_A_VER])
        print(f"\n{'='*60}")
        print(f" 👁️  CONVERSACIÓN #{ENTRADA_A_VER + 1} DE {len(all_lines)}")
        print(f"{'='*60}")

        for msg in entry.get('messages', []):
            role = msg['role'].upper()
            content = msg['content']
            print(f"\n[{role}]")
            print("-" * 40)
            # Para el mensaje del usuario, ocultamos el knowledge (puede ser muy largo)
            if msg['role'] == 'user':
                # Mostramos solo la pregunta (fuera del tag <knowledge>)
                parts = content.split('</knowledge>')
                if len(parts) > 1:
                    knowledge_section = content.split('<knowledge>')[1].split('</knowledge>')[0]
                    question = parts[-1].strip()
                    print(f"<knowledge> [fragmento de {len(knowledge_section.split())} palabras] </knowledge>")
                    print(f"\nPREGUNTA: {question}")
                else:
                    print(content)
            else:
                print(content)
    else:
        print(f"❌ Solo hay {len(all_lines)} entradas. Elige un número entre 0 y {len(all_lines)-1}.")
else:
    print("❌ El dataset no existe aún.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# UTILIDAD: Probar la generación con un texto personalizado
# Útil para verificar que el prompt genera el estilo correcto
# antes de procesar todos los PDFs.
# ─────────────────────────────────────────────────────────────────────────

TEXTO_DE_PRUEBA = """
El tomate (Solanum lycopersicum) requiere condiciones climáticas específicas para su óptimo desarrollo.
La temperatura ideal para el cultivo oscila entre 18 y 26 grados Celsius durante el día.
El exceso de humedad favorece el desarrollo de enfermedades fungosas como la Botrytis cinerea,
conocida como moho gris, que puede devastar una cosecha en pocos días si no se controla a tiempo.
Se recomienda el uso de fungicidas cúpricos en etapas preventivas, especialmente durante los
períodos de alta precipitación. La distancia de siembra recomendada es de 0.5 metros entre plantas
y 1.2 metros entre surcos para garantizar una adecuada aireación y facilitar las labores culturales.
"""

print("🧪 PRUEBA DE GENERACIÓN DE CONVERSACIÓN")
print("-" * 50)
print(f"Texto de entrada ({len(TEXTO_DE_PRUEBA.split())} palabras)")
print("-" * 50)

test_conv, test_skip = generate_conversation(TEXTO_DE_PRUEBA.strip())

if test_conv:
    print("\n✅ Conversación generada exitosamente:")
    print(json.dumps(test_conv, ensure_ascii=False, indent=2))
else:
    print(f"\n⏭️ Chunk omitido. Razón: {test_skip}")

---
## 📋 Referencia rápida

| Parámetro | Descripción | Valor por defecto |
|-----------|-------------|-------------------|
| `OPENAI_API_KEY` | Tu API key de DeepSeek | — |
| `OPENAI_MODEL` | Modelo a usar | `deepseek-chat` |
| `CHUNKS_JSON_FILE` | Ruta al JSON de chunks en Drive | `/content/drive/MyDrive/chunks_agricolas.json` |
| `OUTPUT_FILE` | Archivo de salida `.jsonl` | `/content/drive/MyDrive/dataset_agricola.jsonl` |
| `MAX_RETRIES` | Reintentos si la API falla | `3` |
| `DELAY_SECONDS` | Pausa entre llamadas (segundos) | `1.5` |
| `MODO_INICIO` | `nuevo`, `continuar` o `ampliar` | `nuevo` |

### Campos del JSON de entrada
| Campo | Tipo | Obligatorio | Descripción |
|-------|------|:-----------:|-------------|
| `document_id` | string | ✅ | Nombre o identificador del documento fuente |
| `chunk_number` | número | ✅ | Número del chunk dentro del documento |
| `text` | string | ✅ | Contenido completo del chunk (mínimo 50 palabras) |
| `citation` | string | — | Referencia bibliográfica del documento |
| `date` | string | — | Fecha del documento |

Los campos `citation` y `date` son opcionales: el chunk se procesa igualmente si no están presentes.

### Flujo de uso recomendado
1. Configura los parámetros en la **Sección 2**
2. Ejecuta la **Sección 12 — Utilidad B** con un texto de prueba para verificar el estilo
3. Ejecuta las secciones **1 → 8** en orden para generar el dataset completo
4. Revisa las estadísticas en la **Sección 9**
5. Valida el dataset en la **Sección 10**
6. Descarga los archivos en la **Sección 11**

### Para reanudar o ampliar
- Cambia `MODO_INICIO = "continuar"` para retomar donde quedó tras una interrupción
- Cambia `MODO_INICIO = "ampliar"` para agregar más conversaciones desde un JSON actualizado
- En ambos casos el sistema detecta automáticamente qué chunks ya fueron procesados,
  usando la combinación `document_id + chunk_number` como identificador único